In [ ]:
from pathlib import Path
import tifffile as tf
import numpy as np 
import histomicstk as htk
import matplotlib.pyplot as plt
import skimage
import cv2
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import gaussian_filter

In [ ]:
registered_img_folder_path = Path('Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 2/ANCA 9/1x')

In [ ]:
key = 'ANCA 9'

In [ ]:
color_deconvolution_path = 'Y:/coskun-lab/Aditi/01/Results/color_deconvolution/Multiplex 2/ANCA 9'

In [ ]:
Path.mkdir(Path(color_deconvolution_path), exist_ok=True)

In [ ]:
color_deconvolution_path+= '/' + key 

In [ ]:
Path.mkdir(Path(color_deconvolution_path), exist_ok=True)

In [ ]:
color_images_paths = [f for f in registered_img_folder_path.glob('*tif') if 'Cycle' not in str(f) and 'registered' in str(f)]
color_images_rgb_dict = {}
for path in color_images_paths:

    stain = str(path).split('\\')[-1].split('_')[0]
    print(stain)
    stain_img = tf.imread(path)
    black_pixels_mask = np.all(stain_img == [0, 0, 0], axis=2)
    stain_img[black_pixels_mask] = [np.max(stain_img), np.max(stain_img), np.max(stain_img)]
    color_images_rgb_dict[stain] = stain_img

In [ ]:
with open('/'.join(str(registered_img_folder_path).split('\\')[:-1]) + '/1x/roi_coord_dict.pickle', 'rb') as f:
    roi = pickle.load(f)

In [ ]:
k = 'A'

##### Test htk PAS deconvolution

In [ ]:
PAS_test = color_images_rgb_dict['PAS'][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]]
PAS_test = np.clip(PAS_test * 255, 0, 255).astype(np.uint8)

In [ ]:
w_est_PAS = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(PAS_test, 255)
PAS_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(PAS_test, w_est_PAS, 255)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(PAS_deconvolved.Stains[:, :, 0])
ax[1].imshow(PAS_deconvolved.Stains[:, :, 1])
ax[2].imshow(PAS_deconvolved.Stains[:, :, 2])

##### Test htk Trichrome deconvolution

In [ ]:
Trichrome_test = color_images_rgb_dict['Trichrome'][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]]
Trichrome_test = np.clip(Trichrome_test * 255, 0, 255).astype(np.uint8)

In [ ]:
w_est_Trichrome = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(Trichrome_test, 255)
Trichrome_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome_test, w_est_Trichrome, 255)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(Trichrome_deconvolved.Stains[:, :, 0])
ax[1].imshow(Trichrome_deconvolved.Stains[:, :, 1])
ax[2].imshow(Trichrome_deconvolved.Stains[:, :, 2])

##### Test htk color deconvolution with HE and compare to skimage color deconvolution

In [ ]:
HE_test = color_images_rgb_dict['HE'][roi['A'][0]:roi['A'][1], roi['A'][2]:roi['A'][3]]
HE_test = np.clip(HE_test * 255, 0, 255).astype(np.uint8)

In [ ]:
HE_deconvolved_v1 = skimage.color.rgb2hed(HE_test)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(HE_deconvolved_v1[:, :, 0])
ax[1].imshow(HE_deconvolved_v1[:, :, 1])
ax[2].imshow(HE_deconvolved_v1[:, :, 2])

In [ ]:
w_est_HE = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(HE_test, np.max(HE_test))
HE_deconvolved_v2 = htk.preprocessing.color_deconvolution.color_deconvolution(HE_test, w_est_HE, np.max(HE_test))

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(HE_deconvolved_v2.Stains[:, :, 0])
ax[1].imshow(HE_deconvolved_v2.Stains[:, :, 1])
ax[2].imshow(HE_deconvolved_v2.Stains[:, :, 2])

In [ ]:
stain_color_map = htk.preprocessing.color_deconvolution.stain_color_map
stains = ['hematoxylin', 'eosin','null']         
w_est_HE_given = np.array([stain_color_map[st] for st in stains]).T
HE_deconvolved_v3 = htk.preprocessing.color_deconvolution.color_deconvolution(HE_test, w_est_HE_given)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(HE_deconvolved_v3.Stains[:, :, 0])
ax[1].imshow(HE_deconvolved_v3.Stains[:, :, 1])
ax[2].imshow(HE_deconvolved_v3.Stains[:, :, 2])

##### Compute htk color deconvolution for HE, PAS and Trichrome

In [ ]:
HE = np.clip(color_images_rgb_dict['HE'] * 255, 0, 255).astype(np.uint8)
PAS =  np.clip(color_images_rgb_dict['PAS'] * 255, 0, 255).astype(np.uint8)
Trichrome =  np.clip(color_images_rgb_dict['Trichrome'] * 255, 0, 255).astype(np.uint8)

In [ ]:
w_est_HE_wsi = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(HE, 255)
HE_deconvolved_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(HE, w_est_HE_wsi, 255)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(HE_deconvolved_wsi.Stains[:, :, 0][roi['A'][0]:roi['A'][1], roi['A'][2]:roi['A'][3]])
ax[1].imshow(HE_deconvolved_wsi.Stains[:, :, 1][roi['A'][0]:roi['A'][1], roi['A'][2]:roi['A'][3]])
ax[2].imshow(HE_deconvolved_wsi.Stains[:, :, 2][roi['A'][0]:roi['A'][1], roi['A'][2]:roi['A'][3]])

In [ ]:
w_est_PAS_wsi = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(PAS, 255)
PAS_deconvolved_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(PAS, w_est_PAS_wsi, 255)

In [ ]:
PAS_deconvolved_w_est_non_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(PAS, w_est_PAS, 255)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(PAS_deconvolved_wsi.Stains[:, :, 0][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[1].imshow(PAS_deconvolved_wsi.Stains[:, :, 1][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[2].imshow(PAS_deconvolved_wsi.Stains[:, :, 2][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(PAS_deconvolved_w_est_non_wsi.Stains[:, :, 0][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[1].imshow(PAS_deconvolved_w_est_non_wsi.Stains[:, :, 1][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[2].imshow(PAS_deconvolved_w_est_non_wsi.Stains[:, :, 2][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])

In [ ]:
w_est_Trichrome_wsi = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(Trichrome, 255)
Trichrome_deconvolved_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome, w_est_Trichrome_wsi, 255)

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
k = 'A'
ax[0].imshow(Trichrome_deconvolved_wsi.Stains[:, :, 0][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[1].imshow(Trichrome_deconvolved_wsi.Stains[:, :, 1][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[2].imshow(Trichrome_deconvolved_wsi.Stains[:, :, 2][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])

In [ ]:
Trichrome_deconvolved_w_est_no_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome, w_est_Trichrome, np.max(Trichrome))

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(Trichrome_deconvolved_w_est_no_wsi.Stains[:, :, 0][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[1].imshow(Trichrome_deconvolved_w_est_no_wsi.Stains[:, :, 1][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])
ax[2].imshow(Trichrome_deconvolved_w_est_no_wsi.Stains[:, :, 2][roi[k][0]:roi[k][1], roi[k][2]:roi[k][3]])